# Day 4: MapPLUTOの取得と結合キー検証

**目的**: MapPLUTO区画データとStarbucks店舗のspatial joinが現実的にできるか検証

**判断基準**: 171店中150店以上（88%以上）がマッチ → 問題なし。100以下 → バッファ付きjoin必要

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from scipy.spatial import cKDTree
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium

## 1. MapPLUTO データ概要

In [ ]:
df_pluto = pd.read_csv("../../data/raw/pluto_manhattan.csv", low_memory=False)
print(f"MapPLUTO v25v4 マンハッタン")
print(f"行数: {len(df_pluto):,}, カラム数: {len(df_pluto.columns)}")
print(f"座標充填率: {df_pluto['latitude'].notna().mean()*100:.1f}%")

# 主要カラム
key_cols = ['landuse', 'bldgclass', 'numfloors', 'unitstotal', 'assesstot',
            'yearbuilt', 'bldgarea', 'lotarea', 'retailarea', 'address', 'bbl']
print(f"\n主要カラム:")
for col in key_cols:
    null = df_pluto[col].isnull().sum()
    uniq = df_pluto[col].nunique()
    print(f"  {col:15s}: ユニーク={uniq:6,}, 欠損={null:5,}")

In [ ]:
# LandUse分布
landuse_map = {
    1: 'One & Two Family', 2: 'Multi-Family Walk-Up', 3: 'Multi-Family Elevator',
    4: 'Mixed Res & Commercial', 5: 'Commercial & Office',
    6: 'Industrial', 7: 'Transportation', 8: 'Public Facilities',
    9: 'Open Space', 10: 'Parking', 11: 'Vacant Land'
}

lu_counts = df_pluto['landuse'].value_counts().sort_index()
labels = [landuse_map.get(int(k), str(k)) for k in lu_counts.index if pd.notna(k)]
values = [v for k, v in lu_counts.items() if pd.notna(k)]

fig = go.Figure(go.Bar(x=values, y=labels, orientation='h', marker_color='#3498DB'))
fig.update_layout(title='MapPLUTO マンハッタン LandUse分布',
                  xaxis_title='区画数', template='plotly_white', height=400, width=700)
fig.show()

## 2. Starbucks × PLUTO Nearest Join

In [ ]:
# データ準備
df_p = df_pluto[df_pluto['latitude'].notna()].copy()
df_p['lat'] = pd.to_numeric(df_p['latitude'])
df_p['lon'] = pd.to_numeric(df_p['longitude'])

gdf_sb = gpd.read_file("../../data/raw/osm_starbucks_manhattan.geojson")
gdf_sb['centroid'] = gdf_sb.geometry.centroid
gdf_sb['lat'] = gdf_sb['centroid'].y
gdf_sb['lon'] = gdf_sb['centroid'].x

def latlon_to_meters(lat, lon, ref_lat=40.75):
    y = lat * 111_320
    x = lon * 111_320 * np.cos(np.radians(ref_lat))
    return x, y

pluto_x, pluto_y = latlon_to_meters(df_p['lat'].values, df_p['lon'].values)
sb_x, sb_y = latlon_to_meters(gdf_sb['lat'].values, gdf_sb['lon'].values)

tree = cKDTree(np.column_stack([pluto_x, pluto_y]))
dists, indices = tree.query(np.column_stack([sb_x, sb_y]))

print(f"=== マッチ結果 ===")
for t in [10, 20, 30, 50, 100]:
    n = (dists <= t).sum()
    print(f"  {t:3d}m以内: {n}/171 ({n/171*100:.1f}%)")
print(f"\n中央値: {np.median(dists):.1f}m, 最大: {dists.max():.1f}m")

In [ ]:
# マッチ距離の分布
fig = go.Figure(go.Histogram(x=dists, nbinsx=30, marker_color='#00704A'))
fig.add_vline(x=50, line_dash='dash', line_color='red', annotation_text='50m')
fig.update_layout(title='Starbucks → 最寄りPLUTO区画 距離分布',
                  xaxis_title='距離 (m)', yaxis_title='店舗数',
                  template='plotly_white', width=700, height=350)
fig.show()

## 3. スタバ入居区画の属性分析

In [ ]:
df_matched = df_p.iloc[indices].copy()
df_matched['dist'] = dists

# LandUse
lu = df_matched['landuse'].value_counts()
labels = [landuse_map.get(int(k), str(k)) for k in lu.index if pd.notna(k)]
values = [v for k, v in lu.items() if pd.notna(k)]

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'pie'}, {'type':'bar'}]],
                    subplot_titles=['LandUse分布', '階数分布'])
fig.add_trace(go.Pie(labels=labels, values=values,
                     marker_colors=['#00704A','#D4A76A','#3498DB','#E74C3C','#95A5A6','#F39C12','#8E44AD','#2ECC71']),
              row=1, col=1)

nf = pd.to_numeric(df_matched['numfloors'], errors='coerce').dropna()
fig.add_trace(go.Histogram(x=nf, nbinsx=20, marker_color='#3498DB', name='階数'),
              row=1, col=2)

fig.update_layout(title='スタバ入居ビルの特徴', template='plotly_white',
                  width=900, height=400, showlegend=False)
fig.show()

print(f"階数: 平均{nf.mean():.0f}階, 中央値{nf.median():.0f}階, 最大{nf.max():.0f}階")
print(f"建築年: 平均{pd.to_numeric(df_matched['yearbuilt'], errors='coerce').pipe(lambda s: s[s>1800]).mean():.0f}年")
print(f"小売面積あり: {(pd.to_numeric(df_matched['retailarea'], errors='coerce')>0).sum()}/171")
print(f"評価額中央値: ${pd.to_numeric(df_matched['assesstot'], errors='coerce').median()/1e6:.1f}M")

## 4. 判定結果

### MapPLUTO × Starbucks結合 → **問題なし**

| 項目 | 結果 |
|---|---|
| MapPLUTO バージョン | v25v4（最新） |
| マンハッタン区画数 | 42,600（棚卸し通り） |
| カラム数 | 90 |
| 100m以内マッチ | **171/171 (100%)** |
| 50m以内マッチ | 164/171 (95.9%) |
| 中央値距離 | 21.3m |

### スタバ入居ビルの特徴

| 属性 | 値 |
|---|---|
| LandUse | Commercial & Office 45.6%, Mixed Res/Com 39.2% |
| 平均階数 | 20階（中央値18階） |
| 平均建築年 | 1948年（中央値1942年） |
| 小売面積あり | 87.1% |
| 評価額中央値 | $18.0M |

### 結論

- nearest joinで全171店がPLUTO区画に紐付け可能
- Shapefile(ポリゴン)なしでもポイント座標のnearest joinで十分
- バッファ付きspatial joinへの切り替えは不要